# 🎮 SpriteForge — Генератор спрайт-листов на ИИ

Загрузите изображение персонажа → ИИ создаст позы → Скачайте готовый спрайт-лист

## Как пользоваться

1. Убедитесь, что используется **GPU-рантайм**: `Среда выполнения → Сменить тип среды выполнения → T4 GPU`
2. Запускайте ячейки **сверху вниз**, нажимая кнопку ▶
3. Дождитесь завершения каждой ячейки перед запуском следующей
4. Последняя ячейка скачает спрайт-лист в виде ZIP-архива

---

## Шаг 1: Установка зависимостей
Запустите эту ячейку один раз. Занимает около 2-3 минут.

In [ ]:
# Установка всех зависимостей
!pip install -q diffusers transformers accelerate safetensors
!pip install -q controlnet-aux rembg pillow==11.1.0 opencv-python-headless onnxruntime

# Проверка GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU готов: {gpu_name} ({vram:.1f} ГБ видеопамяти)")
else:
    print("❌ GPU не обнаружен! Перейдите в Среда выполнения → Сменить тип среды выполнения → T4 GPU")

## Шаг 2: Загрузка ИИ-моделей
Первый запуск занимает ~5 минут (скачивается ~4 ГБ). Далее модели кэшируются.

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers import DDIMScheduler
import torch

print("Загрузка ControlNet (управление позами)...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose",
    torch_dtype=torch.float32
)

print("Загрузка Stable Diffusion 1.5...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float32,
    safety_checker=None
)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

print("Загрузка IP-Adapter (сохранение вида персонажа)...")
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)
pipe.set_ip_adapter_scale(0.7)

print("\n✅ Все модели загружены! Готов к генерации спрайтов.")

## Шаг 3: Загрузка персонажа
Загрузите изображение персонажа (PNG или JPG). ИИ удалит фон.

In [ ]:
from google.colab import files
from PIL import Image
from rembg import remove
import numpy as np
import matplotlib.pyplot as plt

print("Выберите файл с изображением персонажа...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Загружено: {filename}")

print("Удаление фона...")
original = Image.open(filename).convert("RGBA")
clean = remove(original)

bbox = clean.getbbox()
if bbox:
    cropped = clean.crop(bbox)
else:
    cropped = clean

canvas = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
max_dim = max(cropped.width, cropped.height)
scale = (512 * 0.8) / max_dim
new_w, new_h = int(cropped.width * scale), int(cropped.height * scale)
resized = cropped.resize((new_w, new_h), Image.LANCZOS)
canvas.paste(resized, ((512 - new_w) // 2, (512 - new_h) // 2), resized)
reference = canvas

small = reference.convert("RGB").resize((64, 64), Image.NEAREST)
unique_colors = len(np.unique(np.array(small).reshape(-1, 3), axis=0))
if unique_colors < 64:
    style = "pixel_art"
elif unique_colors < 512:
    style = "illustration"
else:
    style = "realistic"

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original)
axes[0].set_title("Оригинал")
axes[0].axis("off")
axes[1].imshow(reference)
axes[1].set_title(f"Обработано (стиль: {style})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"\n✅ Персонаж обработан! Стиль: {style}, Уникальных цветов: {unique_colors}")

## Шаг 4: Настройка
Измените параметры для настройки спрайт-листа.

In [ ]:
#@title ⚙️ Настройка { display-mode: "form" }

#@markdown ### Имя персонажа
CHARACTER_NAME = "мой_персонаж"  #@param {type:"string"}

#@markdown ### Размер кадра (пикселей)
FRAME_SIZE = 128  #@param {type:"integer"}

#@markdown ### Набор поз
POSE_PRESET = "basic"  #@param ["minimal", "basic", "full"]

#@markdown ### Формат экспорта
EXPORT_FORMAT = "phaser"  #@param ["generic", "phaser", "godot", "unity", "css"]

#@markdown ### Сид генерации (0 = случайный)
SEED = 12345  #@param {type:"integer"}

#@markdown ---
#@markdown ### Расширенные настройки
IP_SCALE = 0.7  #@param {type:"slider", min:0.3, max:1.0, step:0.05}
CN_SCALE = 0.8  #@param {type:"slider", min:0.3, max:1.0, step:0.05}
STEPS = 25  #@param {type:"slider", min:15, max:50, step:5}

PRESETS = {
    "minimal": ["idle", "walk_1", "attack_1", "death"],
    "basic": ["idle", "walk_1", "walk_2", "attack_1", "jump", "death"],
    "full": ["idle", "walk_1", "walk_2", "run_1", "run_2", "attack_1", "attack_2", "jump", "fall", "hurt", "death", "block"],
}
POSES = PRESETS[POSE_PRESET]

print(f"Персонаж: {CHARACTER_NAME}")
print(f"Позы ({len(POSES)}): {', '.join(POSES)}")
print(f"Размер кадра: {FRAME_SIZE}px")
print(f"Экспорт: {EXPORT_FORMAT}")
print(f"Сид: {SEED if SEED != 0 else 'случайный'}")
print(f"\n✅ Настройка завершена!")

## Шаг 5: Создание скелетов поз
Позы, которые ИИ будет использовать как ориентиры.

In [ ]:
from PIL import ImageDraw

LIMBS = [
    (0,1,"red"),(1,2,"green"),(2,3,"green"),(3,4,"green"),
    (1,5,"blue"),(5,6,"blue"),(6,7,"blue"),(1,8,"yellow"),
    (8,9,"magenta"),(9,10,"magenta"),(10,11,"magenta"),
    (8,12,"cyan"),(12,13,"cyan"),(13,14,"cyan"),
]

def draw_skeleton(kps, size=512):
    img = Image.new("RGB", (size, size), (0,0,0))
    draw = ImageDraw.Draw(img)
    for a,b,c in LIMBS:
        if a < len(kps) and b < len(kps):
            if kps[a]!=(0,0) and kps[b]!=(0,0):
                draw.line([kps[a],kps[b]], fill=c, width=4)
    for x,y in kps:
        if (x,y)!=(0,0):
            draw.ellipse([(x-4,y-4),(x+4,y+4)], fill="white")
    return img

ALL_KEYPOINTS = {
    "idle": [(256,100),(256,160),(200,170),(190,230),(185,280),(312,170),(322,230),(327,280),(256,280),(220,290),(215,360),(210,430),(292,290),(297,360),(302,430)],
    "walk_1": [(256,100),(256,160),(200,170),(180,220),(160,250),(312,170),(330,220),(350,250),(256,280),(230,290),(200,360),(180,430),(282,290),(310,360),(330,430)],
    "walk_2": [(256,100),(256,160),(200,170),(220,220),(240,250),(312,170),(290,220),(270,250),(256,280),(282,290),(310,360),(330,430),(230,290),(200,360),(180,430)],
    "run_1": [(256,90),(256,150),(200,155),(165,130),(140,110),(312,155),(350,180),(370,210),(256,260),(220,270),(180,330),(160,400),(292,270),(330,340),(350,400)],
    "run_2": [(256,90),(256,150),(200,155),(230,180),(250,210),(312,155),(345,130),(370,110),(256,260),(292,270),(330,340),(350,400),(220,270),(180,330),(160,400)],
    "attack_1": [(256,100),(256,160),(200,150),(160,120),(130,90),(312,170),(330,190),(340,220),(256,280),(230,290),(225,360),(220,430),(282,290),(287,360),(292,430)],
    "attack_2": [(270,100),(270,160),(210,160),(180,200),(150,250),(330,170),(350,200),(360,230),(270,280),(240,290),(230,360),(225,430),(300,290),(320,350),(340,410)],
    "jump": [(256,80),(256,140),(200,130),(170,100),(150,80),(312,130),(342,100),(362,80),(256,240),(220,250),(200,290),(210,330),(292,250),(312,290),(302,330)],
    "fall": [(256,120),(256,180),(200,170),(170,140),(150,120),(312,170),(342,140),(362,120),(256,280),(220,290),(210,360),(220,430),(292,290),(302,360),(292,430)],
    "hurt": [(270,110),(265,170),(210,180),(190,220),(175,260),(320,175),(340,210),(350,250),(260,290),(225,300),(215,370),(210,435),(295,295),(310,360),(320,425)],
    "death": [(150,400),(200,400),(160,410),(120,420),(90,430),(240,410),(280,420),(310,430),(280,400),(320,395),(370,400),(420,410),(320,405),(360,415),(400,430)],
    "block": [(256,100),(256,160),(210,160),(220,130),(240,110),(302,160),(292,130),(272,110),(256,280),(220,290),(215,360),(210,430),(292,290),(297,360),(302,430)],
}

pose_images = {}
fig, axes = plt.subplots(1, len(POSES), figsize=(3 * len(POSES), 3))
if len(POSES) == 1:
    axes = [axes]
for i, name in enumerate(POSES):
    skeleton = draw_skeleton(ALL_KEYPOINTS[name])
    pose_images[name] = skeleton
    axes[i].imshow(skeleton)
    axes[i].set_title(name, fontsize=10)
    axes[i].axis("off")
plt.suptitle("Скелеты поз (ориентиры для ИИ)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"✅ Создано скелетов поз: {len(pose_images)}")

## Шаг 6: Генерация поз
ИИ создаёт каждую позу. ~15-30 секунд на позу.

In [ ]:
from rembg import remove as remove_bg
import time

STYLE_PROMPTS = {
    "pixel_art": "pixel art, 2d game sprite, clean pixels, limited palette, retro",
    "illustration": "2d illustration, game character, clean lines, cel shaded",
    "realistic": "detailed character, game asset, clean render, studio lighting",
}

POSE_PROMPTS = {
    "idle": "standing idle, neutral stance, front view",
    "walk_1": "mid-walk, left foot forward, side view",
    "walk_2": "mid-walk, right foot forward, side view",
    "run_1": "running fast, dynamic stride, side view",
    "run_2": "running fast, opposite stride, side view",
    "attack_1": "winding up attack, raising weapon",
    "attack_2": "striking downward, weapon extended",
    "jump": "jumping, feet off ground, arms raised",
    "fall": "falling down, arms flailing",
    "hurt": "recoiling in pain, leaning back",
    "death": "collapsed on ground, defeated",
    "block": "blocking, arms crossed defensively",
}

base_seed = SEED if SEED != 0 else int(time.time()) % (2**32)
generated_poses = {}
style_prompt = STYLE_PROMPTS.get(style, STYLE_PROMPTS["illustration"])

# ВАЖНО: конвертируем в RGB для IP-Adapter
reference_rgb = reference.convert("RGB")

print(f"Генерация {len(POSES)} поз (сид: {base_seed})...\n")
start_time = time.time()

for i, pose_name in enumerate(POSES):
    t0 = time.time()
    print(f"  [{i+1}/{len(POSES)}] Генерация {pose_name}...", end="  ")

    pose_desc = POSE_PROMPTS.get(pose_name, f"character {pose_name} pose")
    prompt = (
        f"character {pose_desc}, {style_prompt}, "
        f"transparent background, single character, centered, game asset"
    )
    negative = (
        "blurry, low quality, deformed, extra limbs, "
        "multiple characters, text, watermark, busy background"
    )

    generator = torch.Generator(device="cpu").manual_seed(base_seed + i)

    result = pipe(
        prompt=prompt,
        negative_prompt=negative,
        image=pose_images[pose_name],
        ip_adapter_image=reference_rgb,
        num_inference_steps=STEPS,
        guidance_scale=7.5,
        controlnet_conditioning_scale=CN_SCALE,
        generator=generator,
        width=512,
        height=512
    ).images[0]

    clean = remove_bg(result)

    bbox = clean.getbbox()
    if bbox:
        cropped = clean.crop(bbox)
        canvas = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
        ox = (512 - cropped.width) // 2
        oy = (512 - cropped.height) // 2
        canvas.paste(cropped, (ox, oy), cropped)
        clean = canvas

    generated_poses[pose_name] = clean
    elapsed = time.time() - t0
    print(f"готово ({elapsed:.1f}с)")

total = time.time() - start_time
print(f"\n✅ Все {len(generated_poses)} поз сгенерированы за {total:.0f}с!")

fig, axes = plt.subplots(1, len(generated_poses), figsize=(3 * len(generated_poses), 3))
if len(generated_poses) == 1:
    axes = [axes]
for i, (name, img) in enumerate(generated_poses.items()):
    axes[i].imshow(img)
    axes[i].set_title(name, fontsize=10)
    axes[i].axis("off")
plt.suptitle("Сгенерированные позы", fontsize=14)
plt.tight_layout()
plt.show()

## Шаг 7: Сборка спрайт-листа
Все позы располагаются в виде сетки.

In [ ]:
import math

n = len(generated_poses)
cols = min(n, 6)
rows = math.ceil(n / cols)
padding = 1
cell = FRAME_SIZE + padding * 2

resample = Image.NEAREST if style == "pixel_art" else Image.LANCZOS

sheet = Image.new("RGBA", (cols * cell, rows * cell), (0, 0, 0, 0))
metadata = []

for i, (name, img) in enumerate(generated_poses.items()):
    r, c = i // cols, i % cols
    x, y = c * cell + padding, r * cell + padding
    frame = img.resize((FRAME_SIZE, FRAME_SIZE), resample)
    sheet.paste(frame, (x, y), frame)
    metadata.append({"name": name, "x": x, "y": y, "w": FRAME_SIZE, "h": FRAME_SIZE})

zoom = 4
zoomed = sheet.resize((sheet.width * zoom, sheet.height * zoom), Image.NEAREST)
plt.figure(figsize=(12, 6))
plt.imshow(zoomed)
plt.title(f"Спрайт-лист — {sheet.width}x{sheet.height} ({FRAME_SIZE}px кадры, {n} поз)", fontsize=14)
plt.axis("off")
plt.show()

print(f"✅ Спрайт-лист собран: {sheet.width}x{sheet.height}, {n} кадров")

## Шаг 8: Экспорт и скачивание
Скачивает ZIP-архив со спрайт-листом и метаданными.

In [ ]:
import json
import os
import zipfile

OUT = f"/tmp/{CHARACTER_NAME}_export"
os.makedirs(f"{OUT}/frames", exist_ok=True)

sheet.save(f"{OUT}/{CHARACTER_NAME}_spritesheet.png")

for name, img in generated_poses.items():
    img.save(f"{OUT}/frames/{name}.png")

if EXPORT_FORMAT == "phaser":
    atlas = {
        "frames": {
            f["name"]: {
                "frame": {"x": f["x"], "y": f["y"], "w": f["w"], "h": f["h"]},
                "rotated": False, "trimmed": False,
                "spriteSourceSize": {"x": 0, "y": 0, "w": f["w"], "h": f["h"]},
                "sourceSize": {"w": f["w"], "h": f["h"]}
            } for f in metadata
        },
        "meta": {
            "app": "SpriteForge",
            "image": f"{CHARACTER_NAME}_spritesheet.png",
            "format": "RGBA8888",
            "size": {"w": sheet.width, "h": sheet.height},
            "scale": "1"
        }
    }
    with open(f"{OUT}/{CHARACTER_NAME}_atlas.json", "w") as f:
        json.dump(atlas, f, indent=2)

elif EXPORT_FORMAT == "godot":
    tres = '[gd_resource type="SpriteFrames" format=3]\n\n'
    tres += f'[ext_resource type="Texture2D" path="res://{CHARACTER_NAME}_spritesheet.png" id="1"]\n\n'
    for f in metadata:
        tres += f'[sub_resource type="AtlasTexture" id="{f["name"]}"]\n'
        tres += 'atlas = ExtResource( "1" )\n'
        tres += f'region = Rect2({f["x"]}, {f["y"]}, {f["w"]}, {f["h"]})\n\n'
    with open(f"{OUT}/{CHARACTER_NAME}_frames.tres", "w") as f:
        f.write(tres)

elif EXPORT_FORMAT == "unity":
    sprites = []
    for f in metadata:
        unity_y = sheet.height - f["y"] - f["h"]
        sprites.append({"name": f["name"], "rect": {"x": f["x"], "y": unity_y, "width": f["w"], "height": f["h"]}, "pivot": {"x": 0.5, "y": 0.0}})
    meta_data = {"fileFormatVersion": 2, "TextureImporter": {"spriteMode": 2, "spritePixelsPerUnit": FRAME_SIZE, "filterMode": 0, "sprites": sprites}}
    with open(f"{OUT}/{CHARACTER_NAME}_spritesheet.png.meta", "w") as f:
        json.dump(meta_data, f, indent=2)

elif EXPORT_FORMAT == "css":
    css = f".{CHARACTER_NAME} {{\n  background-image: url('{CHARACTER_NAME}_spritesheet.png');\n"
    css += f"  width: {FRAME_SIZE}px;\n  height: {FRAME_SIZE}px;\n  image-rendering: pixelated;\n}}\n\n"
    for f in metadata:
        css += f".{CHARACTER_NAME}.{f['name']} {{ background-position: -{f['x']}px -{f['y']}px; }}\n"
    with open(f"{OUT}/{CHARACTER_NAME}_sprite.css", "w") as f:
        f.write(css)

else:
    with open(f"{OUT}/{CHARACTER_NAME}_metadata.json", "w") as f:
        json.dump({"frames": metadata, "meta": {"size": {"w": sheet.width, "h": sheet.height}, "frame_size": FRAME_SIZE}}, f, indent=2)

zip_path = f"/tmp/{CHARACTER_NAME}_sprites.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUT):
        for fname in fnames:
            filepath = os.path.join(root, fname)
            arcname = os.path.relpath(filepath, OUT)
            zf.write(filepath, arcname)

print(f"\nФайлы в архиве:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(f"  {name}")

print(f"\nСкачивание {CHARACTER_NAME}_sprites.zip...")
files.download(zip_path)
print("\n✅ Готово! Спрайт-лист готов к использованию в вашей игре.")

## 🔄 Перегенерировать позу (опционально)
Если поза не нравится — измените имя и сид, затем перезапустите.

In [ ]:
#@title Перегенерировать позу { display-mode: "form" }
REGEN_POSE = "attack_1"  #@param {type:"string"}
REGEN_SEED = 99999  #@param {type:"integer"}

if REGEN_POSE not in POSES:
    print(f"Поза '{REGEN_POSE}' не входит в набор. Доступные: {POSES}")
else:
    pose_desc = POSE_PROMPTS.get(REGEN_POSE, f"character {REGEN_POSE} pose")
    prompt = f"character {pose_desc}, {style_prompt}, transparent background, single character, centered, game asset"
    negative = "blurry, low quality, deformed, extra limbs, multiple characters, text, watermark"
    generator = torch.Generator(device="cpu").manual_seed(REGEN_SEED)

    result = pipe(
        prompt=prompt, negative_prompt=negative,
        image=pose_images[REGEN_POSE], ip_adapter_image=reference_rgb,
        num_inference_steps=STEPS, guidance_scale=7.5,
        controlnet_conditioning_scale=CN_SCALE,
        generator=generator, width=512, height=512
    ).images[0]

    clean = remove_bg(result)
    bbox = clean.getbbox()
    if bbox:
        cropped = clean.crop(bbox)
        canvas = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
        canvas.paste(cropped, ((512-cropped.width)//2, (512-cropped.height)//2), cropped)
        clean = canvas

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(generated_poses[REGEN_POSE])
    axes[0].set_title(f"Старая {REGEN_POSE}")
    axes[0].axis("off")
    axes[1].imshow(clean)
    axes[1].set_title(f"Новая {REGEN_POSE} (сид={REGEN_SEED})")
    axes[1].axis("off")
    plt.show()

    replace = input("Заменить? (y/n): ").strip().lower()
    if replace == "y":
        generated_poses[REGEN_POSE] = clean
        print(f"✅ Поза {REGEN_POSE} заменена! Перезапустите Шаг 7 и 8.")
    else:
        print("Оставлена оригинальная версия.")